# PubMed → Excel
Searches PubMed for each query and exports a formatted `.xlsx` file per query.

uses the free [NCBI E-utilities API](https://www.ncbi.nlm.nih.gov/books/NBK25499/).

**Optional:** Set an NCBI API key as env var `NCBI_API_KEY` for 10 req/s instead of 3 req/s.  
Get a free key at: https://www.ncbi.nlm.nih.gov/account/

In [1]:
# Install dependencies (run once)
%pip install requests pandas openpyxl -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import time
import requests
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

## ⚙️ Configuration — edit here

In [ ]:
QUERIES = [
    "Hippocampus and engram",
    "Susume Tonegawa",
    "Sheena Josselyn",
    # Add more queries here — any PubMed search term works
]

MAX_RESULTS  = 60    # max papers per query
MAX_WORKERS  = 5     # parallel fetches (keep ≤10)
OUTPUT_DIR   = Path(".")  # folder to save .xlsx files
NCBI_API_KEY = os.getenv("NCBI_API_KEY", "")  # optional

## 🔧 API & fetch functions

In [4]:
BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

def _params(extra: dict) -> dict:
    p = {"db": "pubmed", "retmode": "xml", **extra}
    if NCBI_API_KEY:
        p["api_key"] = NCBI_API_KEY
    return p


def search_pmids(query: str) -> list[str]:
    """Return list of PMIDs for a PubMed query."""
    r = requests.get(
        f"{BASE_URL}/esearch.fcgi",
        params=_params({"term": query, "retmax": MAX_RESULTS}),
        timeout=30,
    )
    r.raise_for_status()
    root = ET.fromstring(r.content)
    return [el.text for el in root.findall(".//Id")]


def fetch_paper(pmid: str) -> dict:
    """Fetch all metadata for a single PMID."""
    r = requests.get(
        f"{BASE_URL}/efetch.fcgi",
        params=_params({"id": pmid, "rettype": "abstract"}),
        timeout=30,
    )
    r.raise_for_status()

    root    = ET.fromstring(r.content)
    article = root.find(".//PubmedArticle")
    if article is None:
        return {"pmid": pmid}

    def text(path):
        el = article.find(path)
        return el.text.strip() if el is not None and el.text else ""

    # Title
    title = text(".//ArticleTitle")

    # Abstract — join labelled sections if present
    abstract_els = article.findall(".//AbstractText")
    parts = []
    for ab in abstract_els:
        label = ab.get("Label")
        t = (ab.text or "").strip()
        parts.append(f"{label}: {t}" if label else t)
    abstract = " ".join(parts)

    # Authors
    authors = []
    for a in article.findall(".//Author"):
        last  = (a.findtext("LastName")  or "").strip()
        first = (a.findtext("ForeName") or "").strip()
        if last:
            authors.append(f"{last} {first}".strip())
    authors_str = "; ".join(authors)

    # Journal + date
    journal = text(".//Journal/Title")
    year    = text(".//PubDate/Year") or text(".//PubDate/MedlineDate")[:4]
    month   = text(".//PubDate/Month")
    date    = f"{year}/{month}" if month else year

    # DOI
    doi = ""
    for id_el in article.findall(".//ArticleId"):
        if id_el.get("IdType") == "doi":
            doi = id_el.text.strip()
            break

    return {
        "pmid":     pmid,
        "doi":      doi,
        "journal":  journal,
        "date":     date,
        "authors":  authors_str,
        "title":    title,
        "abstract": abstract,
    }


def fetch_all_papers(pmids: list[str]) -> list[dict]:
    """Fetch papers in parallel, respecting NCBI rate limits."""
    delay   = 0.1 if NCBI_API_KEY else 0.34   # 10/s with key, 3/s without
    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {}
        for pmid in pmids:
            futures[executor.submit(fetch_paper, pmid)] = pmid
            time.sleep(delay)
        for future in as_completed(futures):
            try:
                results.append(future.result())
            except Exception as e:
                print(f"  ✗ PMID {futures[future]}: {e}")
    return results

## 🎨 Excel formatting

In [5]:
HEADER_FILL  = PatternFill("solid", start_color="1F4E79")
HEADER_FONT  = Font(bold=True, color="FFFFFF", name="Arial", size=11)
ROW_FILL_ALT = PatternFill("solid", start_color="EBF2FA")
THIN_BORDER  = Border(bottom=Side(style="thin", color="CCCCCC"))

COL_WIDTHS = {
    "pmid": 12, "doi": 32, "journal": 28,
    "date": 10, "authors": 36, "title": 50, "abstract": 70,
}


def style_sheet(ws):
    columns = list(COL_WIDTHS.keys())

    for col_idx, col_name in enumerate(columns, start=1):
        cell            = ws.cell(row=1, column=col_idx)
        cell.value      = col_name.upper()
        cell.font       = HEADER_FONT
        cell.fill       = HEADER_FILL
        cell.alignment  = Alignment(horizontal="center", vertical="center")
        ws.column_dimensions[get_column_letter(col_idx)].width = COL_WIDTHS[col_name]

    ws.row_dimensions[1].height = 22

    for row_idx in range(2, ws.max_row + 1):
        fill = ROW_FILL_ALT if row_idx % 2 == 0 else None
        for col_idx in range(1, len(columns) + 1):
            cell           = ws.cell(row=row_idx, column=col_idx)
            cell.font      = Font(name="Arial", size=10)
            cell.alignment = Alignment(vertical="top", wrap_text=(col_idx == len(columns)))
            cell.border    = THIN_BORDER
            if fill:
                cell.fill = fill

    ws.freeze_panes   = "A2"
    ws.auto_filter.ref = ws.dimensions


def save_xlsx(records: list[dict], query: str) -> Path:
    safe_name = "".join(c if c.isalnum() or c in " _-" else "_" for c in query).strip()
    path      = OUTPUT_DIR / f"{safe_name}.xlsx"

    df = pd.DataFrame(records, columns=["pmid", "doi", "journal", "date", "authors", "title", "abstract"])
    df.sort_values("date", ascending=False, inplace=True)
    df.to_excel(path, index=False, sheet_name="Papers")

    wb = load_workbook(path)
    style_sheet(wb.active)
    wb.save(path)
    return path

## 🚀 Run

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for query in QUERIES:
    print(f"\n🔍 Searching: {query!r}")
    pmids = search_pmids(query)
    if not pmids:
        print("  No results found.")
        continue
    print(f"  Found {len(pmids)} papers. Fetching details…")
    records = fetch_all_papers(pmids)
    path    = save_xlsx(records, query)
    print(f"  ✓ Saved {len(records)} papers → {path}")


🔍 Searching: 'Anjana Badrinarayanan'
  Found 27 papers. Fetching details…
  ✗ PMID 31147922: 429 Client Error: Too Many Requests for url: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&retmode=xml&id=31147922&rettype=abstract
  ✗ PMID 36074064: 429 Client Error: Too Many Requests for url: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&retmode=xml&id=36074064&rettype=abstract
  ✗ PMID 39256559: 429 Client Error: Too Many Requests for url: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&retmode=xml&id=39256559&rettype=abstract
  ✗ PMID 35635695: 429 Client Error: Too Many Requests for url: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&retmode=xml&id=35635695&rettype=abstract
  ✗ PMID 26240183: 429 Client Error: Too Many Requests for url: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&retmode=xml&id=26240183&rettype=abstract
  ✗ PMID 22753058: 429 Client Error: Too Many Requests for u